In [48]:
import numpy as np
import scikit_posthocs as sp
import pandas as pd
from scipy import stats
from scipy.interpolate import interp1d
from data_store import params
from plot import load_results, load_samples
from itertools import combinations

ALPHA = 0.05  # Significance level for statistical tests

def get_trlse(data_name, raw=False):
    acq_name = "straddle"
    kernel = "matern"
    results = load_results(data_name, acq_name, kernel, TR=True, random=False)
    if raw:
        return results
    samples = load_samples(data_name, acq_name, kernel=kernel)
    starting_point = params[data_name]["n_regions"]
    budget = params[data_name]["budget"]
    t_common = np.linspace(starting_point, budget, 1)  # Define a common timeline
    interp_series = np.asarray(
        [interp1d(t, series, kind='linear', fill_value='extrapolate')(t_common)
                        for t, series in zip(samples, results)]
    )
    return interp_series

def aggregate_results(algorithms, algorithms_results, agg="mean"):
    aggregated = {}
    for algo, results in zip(algorithms, algorithms_results):
        if results is not None:
            if agg == "mean":
                aggregated[algo] = np.mean(results, axis=1)
            elif agg == "last":
                aggregated[algo] = results[:, -1]
            assert len(aggregated[algo]) == 10, f"Expected 10 runs for {algo}, got {len(aggregated[algo])}."
    return aggregated

def friedman_test(df_scores):
    friedman_stat, p_friedman = stats.friedmanchisquare(*[df_scores[col] for col in df_scores.columns])

    print(f"🔬 Friedman Test Results:")
    print(f"   - Statistic: {friedman_stat:.4f}")
    print(f"   - p-value: {p_friedman:.8f}")

    return p_friedman < ALPHA

def run_statistical_analysis(
        algorithms,
        algorithms_results,
        agg="mean"
    ):
    """
    Performs the full statistical analysis for a single dataset.
    1. Aggregates results from files.
    2. Runs the Friedman test.
    3. If significant, runs post-hoc Wilcoxon tests with Holm-Bonferroni correction.
    """
    # --- Step 1: Aggregate Results ---
    # Load the 10 averaged scores for each algorithm into a dictionary.
    aggregated_scores = aggregate_results(algorithms, algorithms_results, agg)

    # Check if we have enough data to compare
    if len(aggregated_scores) < 2:
        print("Not enough valid algorithm results to perform a statistical test.\n")
        return

    # Create a 2D NumPy array from the aggregated scores for the tests.
    # Shape: (num_runs, num_algorithms) -> e.g., (10, 4)
    df_scores = pd.DataFrame(aggregated_scores)
    
    # --- Step 2: Friedman Test ---
    # This test checks if there are any significant differences among the algorithm groups.
    try:
        friedman_test_result = friedman_test(df_scores)

        # --- Step 3: Post-Hoc Tests (if Friedman was significant) ---
        if friedman_test_result:
            print(f"\n✅ The Friedman test is significant (p < {ALPHA}).")
            
            # Perform pairwise Wilcoxon signed-rank tests directly on the DataFrame.
            # The result will be a correctly shaped (4x4) DataFrame of p-values.
            df_melted = df_scores.melt(var_name='algorithm', value_name='score')
            print(df_scores)
            p_values_posthoc = sp.posthoc_wilcoxon(df_melted, val_col='score', group_col='algorithm', p_adjust='holm')

            print("Pairwise Wilcoxon Test Results (Holm-Bonferroni corrected p-values):")
            print(p_values_posthoc.round(4))
            
            # --- FIX: Data-driven interpretation ---
            print("\n💡 Interpretation:")
            # Calculate mean scores to determine the winner
            mean_scores = df_scores.mean()
            
            # Get all unique pairs of algorithms
            algo_pairs = combinations(p_values_posthoc.columns, 2)
            
            significant_pairs_found = False
            for algo1, algo2 in algo_pairs:
                p_val = p_values_posthoc.loc[algo1, algo2]
                if p_val < ALPHA:
                    significant_pairs_found = True
                    # Determine which algorithm has the higher mean score
                    if mean_scores[algo1] > mean_scores[algo2]:
                        winner, loser = algo1, algo2
                    else:
                        winner, loser = algo2, algo1
                    print(f"   - {winner} is SIGNIFICANTLY BETTER than {loser} (p={p_val:.4f})")
            
            if not significant_pairs_found:
                print("   - No pairwise differences were found to be statistically significant after correction.")
            # --- Find and print the single best algorithm ---
            print("\n🏆 Overall Conclusion:")
            top_algo = mean_scores.idxmax()
            is_undisputed_best = True
            
            other_algos = [algo for algo in df_scores.columns if algo != top_algo]
            for other_algo in other_algos:
                # Check if the top algorithm is significantly better than all others
                p_val = p_values_posthoc.loc[top_algo, other_algo]
                if p_val >= ALPHA:
                    is_undisputed_best = False
                    break # No need to check further
            
            if is_undisputed_best:
                print(f"   - The best performing algorithm is ✨ {top_algo} ✨.")
                print(f"   - It was statistically superior to all other algorithms.")
            else:
                print(f"   - There is no single, undisputed best algorithm.")
                print(f"   - {top_algo} had the highest average score, but was not statistically superior to all others.")


        else:
            print(f"\n❌ The Friedman test is not significant (p >= {ALPHA}).")
            print("   There is no statistical evidence of a difference among the algorithms.\n")

    except ValueError as e:
        print(f"An error occurred during the statistical test: {e}")
        raise e
    
    print("\n")

def dataset_test(data_name):
    print("-" * 70)
    print(f"📊 Analyzing Dataset: {data_name}")
    print("-" * 70)
    trlse_results = get_trlse(data_name, raw=True)
    hlse_results = load_results(data_name, "HLSE", kernel="", TR=False)
    simple_results = load_results(data_name, "straddle", kernel="matern", TR=False)
    random_results = load_results(data_name, "random", kernel="matern", TR=False)
    print("Mean aggregation")
    run_statistical_analysis(
        ["TRLSE", "Random", "Straddle", "HLSE"],
        [trlse_results, random_results, simple_results, hlse_results],
        "mean"
    )
    print("Last aggregation")
    run_statistical_analysis(
        ["TRLSE", "Random", "Straddle", "HLSE"],
        [trlse_results, random_results, simple_results, hlse_results],
        "last"
    )

In [50]:
for data_name in ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]:
    dataset_test(data_name)

----------------------------------------------------------------------
📊 Analyzing Dataset: levy10D
----------------------------------------------------------------------
Mean aggregation
🔬 Friedman Test Results:
   - Statistic: 28.0800
   - p-value: 0.00000349

✅ The Friedman test is significant (p < 0.05).
      TRLSE    Random  Straddle      HLSE
0  0.530465  0.005527  0.013617  0.417311
1  0.536534  0.004593  0.017784  0.420651
2  0.527838  0.014854  0.019801  0.328161
3  0.531564  0.003485  0.017763  0.392062
4  0.519966  0.009875  0.007328  0.492508
5  0.528702  0.005837  0.012163  0.467714
6  0.548095  0.005500  0.014625  0.440631
7  0.521133  0.011619  0.009522  0.504733
8  0.543846  0.008020  0.019769  0.363578
9  0.543013  0.009980  0.012439  0.340689
Pairwise Wilcoxon Test Results (Holm-Bonferroni corrected p-values):
           TRLSE  Random  Straddle    HLSE
TRLSE     1.0000  0.0117    0.0117  0.0117
Random    0.0117  1.0000    0.0137  0.0117
Straddle  0.0117  0.0137    1.